# Day 2: Data Cleaning, NAV Resampling, & DB Loading

In [1]:
import os
import sqlite3
from pathlib import Path
import pandas as pd
import numpy as np
from sqlalchemy import create_engine

# Set directories
raw_dir = Path('../data/raw')
processed_dir = Path('../data/processed')
processed_dir.mkdir(parents=True, exist_ok=True)

db_dir = Path('../data/db')
db_dir.mkdir(parents=True, exist_ok=True)
db_file = db_dir / "bluestock_mf.db"

print(f"Cleaned outputs destination: {processed_dir.resolve()}")
print(f"Target SQLite database path: {db_file.resolve()}")

Cleaned outputs destination: E:\MutualFundAnalysis\data\processed
Target SQLite database path: E:\MutualFundAnalysis\data\db\bluestock_mf.db


## 1. Fund Master Cleaning

In [2]:
fm = pd.read_csv(raw_dir / "01_fund_master.csv")
print(f"Initial shape of Fund Master: {fm.shape}")
fm = fm.drop_duplicates()
for col in ['fund_house', 'scheme_name', 'category', 'sub_category', 'plan', 'benchmark', 'fund_manager', 'risk_category']:
    if col in fm.columns:
        fm[col] = fm[col].astype(str).str.strip()
fm.to_csv(processed_dir / "clean_fund_master.csv", index=False)
print(f"Cleaned Fund Master shape: {fm.shape}")
fm.head(3)

Initial shape of Fund Master: (40, 15)
Cleaned Fund Master shape: (40, 15)


,amfi_code,fund_house,scheme_name,category,sub_category,plan,launch_date,benchmark,expense_ratio_pct,exit_load_pct,min_sip_amount,min_lumpsum_amount,fund_manager,risk_category,sebi_category_code
0,119551,SBI Mutual Fund,SBI Bluechip Fund - Regular Plan - Growth,Equity,Large Cap,Regular,2006-02-14,NIFTY 100 TRI,1.54,1.0,500,1000,Sohini Andani,Moderate,EC01
1,119552,SBI Mutual Fund,SBI Bluechip Fund - Direct Plan - Growth,Equity,Large Cap,Direct,2013-01-01,NIFTY 100 TRI,0.66,1.0,500,1000,Sohini Andani,Moderate,EC01
2,119598,SBI Mutual Fund,SBI Small Cap Fund - Regular Plan - Growth,Equity,Small Cap,Regular,2009-09-09,BSE 250 SmallCap TRI,1.43,1.0,500,1000,R. Srinivasan,Very High,EC03


## 2. Cleaning and Forward-Filling NAV Price History

In [3]:
nav = pd.read_csv(raw_dir / "02_nav_history.csv")
print(f"Initial NAV records count: {nav.shape[0]}")
nav['date'] = pd.to_datetime(nav['date'])
nav = nav.drop_duplicates(subset=['amfi_code', 'date'])

all_dates = pd.date_range(start=nav['date'].min(), end=nav['date'].max(), freq='D')
cleaned_nav_list = []

# Forward-fill per mutual fund scheme group
for code, group in nav.groupby('amfi_code'):
    group = group.set_index('date').reindex(all_dates)
    group['amfi_code'] = code
    group['nav'] = group['nav'].ffill()
    group = group.dropna(subset=['nav'])
    group = group.reset_index().rename(columns={'index': 'date'})
    group = group.sort_values('date')
    group['daily_return_pct'] = group['nav'].pct_change() * 100
    group['daily_return_pct'] = group['daily_return_pct'].fillna(0.0)
    cleaned_nav_list.append(group)

clean_nav = pd.concat(cleaned_nav_list).sort_values(by=['amfi_code', 'date'])
clean_nav.to_csv(processed_dir / "clean_nav.csv", index=False)
print(f"Resampled and Cleaned NAV records: {clean_nav.shape[0]}")
clean_nav.head(5)

Initial NAV records count: 46000


Resampled and Cleaned NAV records: 64320


,date,amfi_code,nav,daily_return_pct
0,2022-01-03,100016,520.4608,0.000000
1,2022-01-04,100016,515.0971,-1.030568
2,2022-01-05,100016,521.7239,1.286515
3,2022-01-06,100016,515.7880,-1.137747
4,2022-01-07,100016,515.1639,-0.120999


## 3. Cleaning Auxiliary Fact Datasets

In [4]:
aux_files = [
    ("03_aum_by_fund_house.csv", "clean_aum_by_fund_house.csv"),
    ("04_monthly_sip_inflows.csv", "clean_monthly_sip_inflows.csv"),
    ("05_category_inflows.csv", "clean_category_inflows.csv"),
    ("06_industry_folio_count.csv", "clean_industry_folio_count.csv"),
    ("09_portfolio_holdings.csv", "clean_portfolio_holdings.csv"),
    ("10_benchmark_indices.csv", "clean_benchmark_indices.csv"),
]

for filename, name in aux_files:
    df = pd.read_csv(raw_dir / filename)
    date_col = 'date' if 'date' in df.columns else 'month' if 'month' in df.columns else 'portfolio_date'
    df[date_col] = df[date_col].astype(str).str.strip()
    for col in df.select_dtypes(include='object').columns:
        df[col] = df[col].astype(str).str.strip()
    df = df.drop_duplicates()
    df.to_csv(processed_dir / name, index=False)
    print(f"Cleaned {filename} -> {name} | shape: {df.shape}")

Cleaned 03_aum_by_fund_house.csv -> clean_aum_by_fund_house.csv | shape: (90, 5)
Cleaned 04_monthly_sip_inflows.csv -> clean_monthly_sip_inflows.csv | shape: (48, 6)
Cleaned 05_category_inflows.csv -> clean_category_inflows.csv | shape: (144, 3)
Cleaned 06_industry_folio_count.csv -> clean_industry_folio_count.csv | shape: (21, 6)
Cleaned 09_portfolio_holdings.csv -> clean_portfolio_holdings.csv | shape: (322, 8)
Cleaned 10_benchmark_indices.csv -> clean_benchmark_indices.csv | shape: (8050, 3)


C:\Users\raksh\AppData\Local\Temp\ipykernel_10028\2184359158.py:14: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include='object').columns:
C:\Users\raksh\AppData\Local\Temp\ipykernel_10028\2184359158.py:14: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guid

## 4. Cleaning Performance & Transactions

In [5]:
perf = pd.read_csv(raw_dir / "07_scheme_performance.csv")
for col in ['scheme_name', 'fund_house', 'category', 'plan', 'risk_grade']:
    if col in perf.columns:
        perf[col] = perf[col].astype(str).str.strip()
perf['negative_sharpe'] = perf['sharpe_ratio'] < 0
perf = perf.drop_duplicates()
perf.to_csv(processed_dir / "clean_performance.csv", index=False)
print(f"Cleaned Performance shape: {perf.shape}")

tx = pd.read_csv(raw_dir / "08_investor_transactions.csv")
tx['transaction_date'] = pd.to_datetime(tx['transaction_date'])
for col in ['transaction_type', 'kyc_status', 'state', 'city', 'city_tier', 'age_group', 'gender', 'payment_mode']:
    if col in tx.columns:
        tx[col] = tx[col].astype(str).str.strip()
tx = tx[tx['amount_inr'] > 0]
tx = tx.drop_duplicates()
tx.to_csv(processed_dir / "clean_transactions.csv", index=False)
print(f"Cleaned Transactions shape: {tx.shape}")

Cleaned Performance shape: (40, 20)


Cleaned Transactions shape: (32778, 13)


## 5. SQL Schema Initialization

In [6]:
if db_file.exists():
    os.remove(db_file)

engine = create_engine(f"sqlite:///{db_file}")

with sqlite3.connect(db_file) as conn:
    with open("../sql/schema.sql", "r") as f:
        schema_sql = f.read()
        conn.executescript(schema_sql)
print("Database schema successfully generated.")

Database schema successfully generated.


## 6. Dynamic Date Dimension & Database Loading

In [7]:
dfs = {
    "dim_fund": pd.read_csv(processed_dir / "clean_fund_master.csv"),
    "fact_nav": pd.read_csv(processed_dir / "clean_nav.csv"),
    "fact_transactions": pd.read_csv(processed_dir / "clean_transactions.csv"),
    "fact_performance": pd.read_csv(processed_dir / "clean_performance.csv"),
    "fact_aum": pd.read_csv(processed_dir / "clean_aum_by_fund_house.csv"),
    "fact_portfolio": pd.read_csv(processed_dir / "clean_portfolio_holdings.csv"),
    "fact_benchmarks": pd.read_csv(processed_dir / "clean_benchmark_indices.csv"),
    "fact_sip_industry": pd.read_csv(processed_dir / "clean_monthly_sip_inflows.csv"),
    "fact_category_inflows": pd.read_csv(processed_dir / "clean_category_inflows.csv"),
    "fact_folio_count": pd.read_csv(processed_dir / "clean_industry_folio_count.csv")
}

nav_dates = pd.to_datetime(dfs["fact_nav"]["date"])
tx_dates = pd.to_datetime(dfs["fact_transactions"]["transaction_date"])
bench_dates = pd.to_datetime(dfs["fact_benchmarks"]["date"])

unique_dates = pd.concat([nav_dates, tx_dates, bench_dates]).dropna().unique()
date_df = pd.DataFrame({"date": unique_dates})
date_df["date_id"] = date_df["date"].dt.strftime("%Y-%m-%d")
date_df["year"] = date_df["date"].dt.year
date_df["month"] = date_df["date"].dt.month
date_df["quarter"] = date_df["date"].dt.quarter
date_df["is_weekday"] = date_df["date"].dt.dayofweek.isin(range(5)).astype(int)

dim_date = date_df[["date_id", "date", "year", "month", "quarter", "is_weekday"]].drop_duplicates().sort_values("date_id")

# Append records
with engine.connect() as conn:
    dim_date.to_sql("dim_date", conn, if_exists="append", index=False)
    dfs["dim_fund"].to_sql("dim_fund", conn, if_exists="append", index=False)
    for table_name, df in dfs.items():
        if table_name != "dim_fund":
            df.to_sql(table_name, conn, if_exists="append", index=False)

print("All cleaned datasets and generated tables loaded successfully.")

All cleaned datasets and generated tables loaded successfully.


## 7. Row Counts and Database Verification

In [8]:
with sqlite3.connect(db_file) as conn:
    tables = pd.read_sql_query("SELECT name FROM sqlite_master WHERE type='table';", conn)
    print("Tables list and row statistics:")
    for table in tables['name']:
        if table != 'sqlite_sequence':
            count = pd.read_sql_query(f"SELECT COUNT(*) as cnt FROM {table}", conn)['cnt'].iloc[0]
            print(f" - {table}: {count} rows")

Tables list and row statistics:
 - dim_fund: 40 rows
 - dim_date: 1608 rows
 - fact_nav: 64320 rows
 - fact_transactions: 32778 rows
 - fact_performance: 40 rows
 - fact_aum: 90 rows
 - fact_portfolio: 322 rows
 - fact_benchmarks: 8050 rows
 - fact_sip_industry: 48 rows
 - fact_category_inflows: 144 rows
 - fact_folio_count: 21 rows
